In [ ]:
import pandas as pd
import geopandas as gpd
from scipy.spatial import cKDTree
import requests
import time
import os
!pip install python-dotenv
from dotenv import load_dotenv

load_dotenv()

In [ ]:
onemap_api_key = os.environ.get('ONEMAP_API_KEY')

if not onemap_api_key:
    raise ValueError(
        "ONEMAP_API_KEY not found. Please create a .env file in this directory with: \n"
        "ONEMAP_API_KEY=your_api_key_here"
    )

In [ ]:
# Get file names 
hdb_points_file = "resale_2019_to_2025_cleaned.geojson"
mrt_exits_file = "LTAMRTStationExitGEOJSON.geojson"
output_file = "hdb_mrt_distances.csv"

### Run a sample of postal codes to ensure code works, check the result output to see if there are any nulls 

In [ ]:
# --- 1. load and prep data ---
raw_hdb_gdf = gpd.read_file(hdb_points_file).to_crs(epsg=4326)
# take only the first 5 unique postals for this sample
postal_gdf_sample = raw_hdb_gdf.drop_duplicates(subset=['postal_code']).head(5).copy()

mrt_exits_gdf = gpd.read_file(mrt_exits_file).to_crs(epsg=4326)

mrt_coords = list(zip(mrt_exits_gdf.geometry.y, mrt_exits_gdf.geometry.x))
mrt_names = mrt_exits_gdf['STATION_NA'].tolist() 
mrt_exit_codes = mrt_exits_gdf['EXIT_CODE'].tolist() 
tree = cKDTree(mrt_coords)

results = []

print(f"--- starting sample test for {len(postal_gdf_sample)} postals ---")

# --- 2. sample loop ---
for idx, row in postal_gdf_sample.iterrows():
    p_code = str(row['postal_code'])
    origin_coords = (row.geometry.y, row.geometry.x)
    
    print(f"\nchecking postal: {p_code} at {origin_coords}")
    
    # get 3 closest mrt exits by straight line
    dist, indices = tree.query(origin_coords, k=3) 
    
    best_walk = float('inf')
    closest_mrt = None
    closest_exit = None

    for i in indices:
        dest_coords = mrt_coords[i]
        mrt_name = mrt_names[i]
        mrt_exit = mrt_exit_codes[i]
        
        # using the routingsvc endpoint from the docs
        url = f"https://www.onemap.gov.sg/api/public/routingsvc/route?start={origin_coords[0]},{origin_coords[1]}&end={dest_coords[0]},{dest_coords[1]}&routeType=walk"
        
        try:

            token = onemap_api_key if onemap_api_key.startswith("Bearer ") else f"Bearer {onemap_api_key}"
            headers = {"Authorization": token}
            res = requests.get(url, headers=headers, timeout=10)
            
            if res.status_code == 200:
                data = res.json()
                if 'route_summary' in data:
                    walk_m = data['route_summary']['total_distance']
                    print(f"  -> found route to {mrt_name}: {walk_m}m")
                    if walk_m < best_walk:
                        best_walk = walk_m
                        closest_mrt = mrt_name
                        closest_exit = mrt_exit
                else:
                    print(f"  -> no route summary found for {mrt_name}")
            else:
                print(f"  -> api error {res.status_code}: {res.text}")
            
            time.sleep(0.2) 
        except Exception as e:
            print(f"  -> request failed: {e}")

    results.append({
        "postal_code": p_code,
        "nearest_mrt": closest_mrt,
        "nearest_mrt_exit": closest_exit,
        "walking_dist_m": best_walk if best_walk != float('inf') else None
    })

# --- 3. display output ---
sample_df = pd.DataFrame(results)
print("\n--- final sample output ---")
print(sample_df)

# check if we actually got results
if sample_df['walking_dist_m'].isnull().all():
    print("\nwarning: all walking distances are null. check your api key and coordinate order.")
else:
    print("\nsuccess: distances found! you can now run the full script.")

### Run full script if results generated for the sample postal codes

In [ ]:
# --- 1. Load Datasets (full) ---
raw_hdb_gdf = gpd.read_file(hdb_points_file).to_crs(epsg=4326)
postal_gdf = raw_hdb_gdf.drop_duplicates(subset=['postal_code']).copy()

mrt_exits_gdf = gpd.read_file(mrt_exits_file).to_crs(epsg=4326)

mrt_coords = list(zip(mrt_exits_gdf.geometry.y, mrt_exits_gdf.geometry.x))
mrt_names = mrt_exits_gdf['STATION_NA'].tolist() 
mrt_exit_codes = mrt_exits_gdf['EXIT_CODE'].tolist() 
tree = cKDTree(mrt_coords)

# --- 2. Resume logic ---
if os.path.exists(output_file):
    processed_df = pd.read_csv(output_file)
    processed_postals = set(processed_df['postal_code'].astype(str))
    results = processed_df.to_dict('records')
    print(f"Resuming... {len(processed_postals)} postals already completed.")
else:
    results = []
    processed_postals = set()

# --- 3. Full loop ---
print(f"Starting processing for {len(postal_gdf) - len(processed_postals)} remaining postals...")

try:
    for idx, row in postal_gdf.iterrows():
        p_code = row['postal_code']
        if p_code in processed_postals:
            continue

        origin_coords = (row.geometry.y, row.geometry.x)
        dist, indices = tree.query(origin_coords, k=3)
        
        best_walk = float('inf')
        best_mrt = None
        best_mrt_exit = None

        for i in indices:
            dest_coords = mrt_coords[i]
            token = onemap_api_key if onemap_api_key.startswith("Bearer ") else f"Bearer {onemap_api_key}"
            url = f"https://www.onemap.gov.sg/api/public/routingsvc/route?start={origin_coords[0]},{origin_coords[1]}&end={dest_coords[0]},{dest_coords[1]}&routeType=walk"
            
            try:
                res = requests.get(url, headers={"Authorization": token}, timeout=10)
                if res.status_code == 200:
                    data = res.json()
                    if 'route_summary' in data:
                        walk_m = data['route_summary']['total_distance']
                        if walk_m < best_walk:
                            best_walk = walk_m
                            best_mrt = mrt_names[i]
                            best_mrt_exit = mrt_exit_codes[i]
                time.sleep(0.2)
            except:
                continue

        results.append({
            "postal_code": p_code,
            "nearest_mrt": best_mrt,
            "nearest_exit": best_mrt_exit,
            "walking_dist_m": best_walk if best_walk != float('inf') else None
        })
        processed_postals.add(p_code)

        if len(results) % 50 == 0:
            pd.DataFrame(results).to_csv(output_file, index=False)
            print(f"progress: {len(results)} / {len(postal_gdf)} unique postals saved.")
            

except KeyboardInterrupt:
    print("\nPaused. Saving current progress...")

# Final save
pd.DataFrame(results).to_csv(output_file, index=False)
print(f"Task complete! Results saved to {output_file}")

In [ ]:
df = pd.read_csv("hdb_mrt_distances.csv")
len(df)

### Checking for any anomalies, then verify on onemap if walking distance is too big

In [ ]:
df[df["walking_dist_m"] > 2500]